# Mixture of Experts (MoE) from Scratch

Mixture of Experts: scale model capacity without proportionally scaling compute by activating only a subset of parameters per token.

**Interview questions:**
- "Why MoE?"
- "What is expert collapse and how do you prevent it?"
- "Active vs total parameters?"
- "How does the router/gating work?"

---

## Key Idea

Replace the dense FFN in each transformer block with a **sparse** MoE layer:

$$\text{MoE}(x) = \sum_{i \in \text{TopK}} g_i(x) \cdot E_i(x)$$

where $g_i(x)$ are gating weights from a router, and $E_i$ are expert networks.

| Model | Total Params | Active Params | Experts | Top-K |
|-------|-------------|--------------|---------|-------|
| Mixtral 8x7B | 47B | ~13B | 8 | 2 |
| Switch Transformer | varies | varies | 128+ | 1 |
| GShard | 600B | ~10B | 2048 | 2 |
| DeepSeek-V2 | 236B | ~21B | 160 | 6 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Expert Network

Each expert is a standard FFN (same architecture as the dense transformer FFN).

In [ ]:
class Expert(nn.Module):
    """Single expert: a standard feed-forward network."""
    def __init__(self, d_model: int, d_ff: int = None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# Test single expert
d_model = 256
expert = Expert(d_model)
x = torch.randn(2, 16, d_model)
out = expert(x)
print(f"Expert params: {sum(p.numel() for p in expert.parameters()):,}")
print(f"Output shape: {out.shape}")

## 2. Router / Gating Network

The router decides which experts to activate for each token. It produces a probability distribution over experts.

$$g(x) = \text{softmax}(W_g \cdot x)$$

Then select top-K experts and renormalize their weights.

In [ ]:
class Router(nn.Module):
    """Top-K router for Mixture of Experts.
    
    For each token, selects top_k experts out of n_experts.
    Returns gating weights (normalized) and expert indices.
    """
    def __init__(self, d_model: int, n_experts: int, top_k: int = 2,
                 noise_std: float = 0.1):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.noise_std = noise_std
        
        self.gate = nn.Linear(d_model, n_experts, bias=False)
    
    def forward(self, x: torch.Tensor) -> tuple:
        """Route tokens to experts.
        
        Args:
            x: (batch * seq_len, d_model) -- flattened token representations
        
        Returns:
            gate_weights: (batch * seq_len, top_k) -- normalized weights for selected experts
            expert_indices: (batch * seq_len, top_k) -- which experts are selected
            router_logits: (batch * seq_len, n_experts) -- raw logits for load balancing
        """
        # Compute logits
        logits = self.gate(x)  # (N, n_experts)
        
        # Add noise during training for exploration
        if self.training and self.noise_std > 0:
            noise = torch.randn_like(logits) * self.noise_std
            logits = logits + noise
        
        # Full softmax over all experts
        router_probs = F.softmax(logits, dim=-1)  # (N, n_experts)
        
        # Select top-K experts
        gate_weights, expert_indices = torch.topk(router_probs, self.top_k, dim=-1)
        # (N, top_k) each
        
        # Renormalize top-K weights to sum to 1
        gate_weights = gate_weights / gate_weights.sum(dim=-1, keepdim=True)
        
        return gate_weights, expert_indices, logits


# Test router
router = Router(d_model=256, n_experts=8, top_k=2)
tokens = torch.randn(32, 256)  # 32 tokens
weights, indices, logits = router(tokens)

print(f"Gate weights shape: {weights.shape} (32 tokens, top-2 weights)")
print(f"Expert indices shape: {indices.shape} (32 tokens, top-2 expert ids)")
print(f"\nSample token 0:")
print(f"  Selected experts: {indices[0].tolist()}")
print(f"  Gate weights: {weights[0].tolist()}")
print(f"  Sum of weights: {weights[0].sum().item():.4f}")

## 3. MoE Layer with Top-K Routing

The full MoE layer: router selects experts, each token is processed by its selected experts, outputs are combined with gating weights.

In [ ]:
class MoELayer(nn.Module):
    """Mixture of Experts layer.
    
    Replaces the dense FFN in a transformer block.
    Each token is routed to top_k experts.
    """
    def __init__(self, d_model: int, d_ff: int, n_experts: int, top_k: int = 2,
                 noise_std: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_experts = n_experts
        self.top_k = top_k
        
        self.router = Router(d_model, n_experts, top_k, noise_std)
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(n_experts)])
    
    def forward(self, x: torch.Tensor) -> tuple:
        """Forward pass.
        
        Args:
            x: (batch, seq_len, d_model)
        
        Returns:
            output: (batch, seq_len, d_model)
            router_logits: for computing load balancing loss
        """
        B, T, D = x.shape
        
        # Flatten batch and sequence dims
        x_flat = x.view(-1, D)  # (N, D) where N = B*T
        N = x_flat.shape[0]
        
        # Route
        gate_weights, expert_indices, router_logits = self.router(x_flat)
        # gate_weights: (N, top_k), expert_indices: (N, top_k)
        
        # Compute expert outputs
        output = torch.zeros_like(x_flat)  # (N, D)
        
        # Process each expert: gather tokens assigned to it, process, scatter back
        for k in range(self.top_k):
            for e_idx in range(self.n_experts):
                # Find tokens assigned to this expert at position k
                mask = (expert_indices[:, k] == e_idx)  # (N,)
                if mask.any():
                    expert_input = x_flat[mask]  # (n_tokens, D)
                    expert_output = self.experts[e_idx](expert_input)  # (n_tokens, D)
                    # Weight by gating coefficient
                    w = gate_weights[mask, k].unsqueeze(-1)  # (n_tokens, 1)
                    output[mask] += w * expert_output
        
        return output.view(B, T, D), router_logits
    
    @property
    def total_params(self):
        return sum(p.numel() for p in self.parameters())
    
    @property
    def active_params_per_token(self):
        """Parameters activated per token (top_k experts + router)."""
        expert_params = sum(p.numel() for p in self.experts[0].parameters())
        router_params = sum(p.numel() for p in self.router.parameters())
        return self.top_k * expert_params + router_params


# Test MoE layer
moe = MoELayer(d_model=256, d_ff=512, n_experts=8, top_k=2)
x = torch.randn(2, 16, 256)
out, router_logits = moe(x)

print(f"MoE layer:")
print(f"  Total params: {moe.total_params:,}")
print(f"  Active params per token: {moe.active_params_per_token:,}")
print(f"  Ratio (active/total): {moe.active_params_per_token/moe.total_params*100:.1f}%")
print(f"  Output shape: {out.shape}")

## 4. Load Balancing Loss

Without regularization, the router tends to collapse to always selecting the same 1-2 experts ("expert collapse"). The **auxiliary load balancing loss** encourages uniform expert utilization.

$$\mathcal{L}_{\text{balance}} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$

where:
- $f_i$ = fraction of tokens routed to expert $i$
- $P_i$ = average router probability for expert $i$
- $N$ = number of experts
- $\alpha$ = balancing coefficient (typically 0.01)

In [ ]:
def load_balancing_loss(router_logits: torch.Tensor, expert_indices: torch.Tensor,
                        n_experts: int, top_k: int) -> torch.Tensor:
    """Compute auxiliary load balancing loss.
    
    Encourages uniform routing across all experts.
    
    Args:
        router_logits: (N, n_experts) raw router logits
        expert_indices: (N, top_k) selected expert indices
        n_experts: total number of experts
        top_k: number of experts per token
    
    Returns:
        loss: scalar balancing loss
    """
    N = router_logits.shape[0]
    
    # f_i: fraction of tokens dispatched to each expert
    # One-hot encode expert assignments and average
    expert_mask = F.one_hot(expert_indices, n_experts).float()  # (N, top_k, n_experts)
    expert_mask = expert_mask.sum(dim=1)  # (N, n_experts) -- how many times each expert selected
    f = expert_mask.mean(dim=0)  # (n_experts,) -- fraction of tokens per expert
    
    # P_i: average router probability for each expert
    router_probs = F.softmax(router_logits, dim=-1)  # (N, n_experts)
    P = router_probs.mean(dim=0)  # (n_experts,)
    
    # Load balancing loss
    loss = n_experts * (f * P).sum()
    
    return loss


# Demo: balanced vs unbalanced routing
n_experts = 8
N = 1000

# Balanced: all experts equally likely
balanced_logits = torch.randn(N, n_experts) * 0.1
_, balanced_indices = torch.topk(F.softmax(balanced_logits, dim=-1), 2, dim=-1)
balanced_loss = load_balancing_loss(balanced_logits, balanced_indices, n_experts, 2)

# Unbalanced: expert 0 strongly preferred
unbalanced_logits = torch.randn(N, n_experts) * 0.1
unbalanced_logits[:, 0] += 5.0  # bias toward expert 0
_, unbalanced_indices = torch.topk(F.softmax(unbalanced_logits, dim=-1), 2, dim=-1)
unbalanced_loss = load_balancing_loss(unbalanced_logits, unbalanced_indices, n_experts, 2)

print(f"Balanced routing loss: {balanced_loss.item():.4f}")
print(f"Unbalanced routing loss: {unbalanced_loss.item():.4f}")
print(f"\nHigher loss = more imbalanced (penalty for non-uniform routing)")

# Visualize expert utilization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, indices, title in [(axes[0], balanced_indices, 'Balanced'),
                            (axes[1], unbalanced_indices, 'Unbalanced')]:
    counts = torch.zeros(n_experts)
    for e in range(n_experts):
        counts[e] = (indices == e).sum().item()
    counts = counts / counts.sum()
    
    ax.bar(range(n_experts), counts.numpy(), color='steelblue', edgecolor='white')
    ax.axhline(y=1/n_experts, color='red', linestyle='--', label=f'Ideal: {1/n_experts:.2f}')
    ax.set_xlabel('Expert ID')
    ax.set_ylabel('Fraction of Tokens')
    ax.set_title(f'{title} Routing')
    ax.legend()
    ax.set_ylim(0, max(counts.max().item() * 1.2, 0.3))

plt.tight_layout()
plt.show()

## 5. Expert Capacity and Token Dropping

In practice, each expert has a fixed **capacity** -- the maximum number of tokens it can process. Tokens that exceed capacity are **dropped** (processed by a residual connection instead).

$$\text{capacity} = \text{capacity\_factor} \times \frac{N \times \text{top\_k}}{E}$$

where $N$ = number of tokens, $E$ = number of experts.

In [ ]:
class MoELayerWithCapacity(nn.Module):
    """MoE layer with expert capacity limits and token dropping."""
    
    def __init__(self, d_model: int, d_ff: int, n_experts: int, top_k: int = 2,
                 capacity_factor: float = 1.25, noise_std: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_experts = n_experts
        self.top_k = top_k
        self.capacity_factor = capacity_factor
        
        self.router = Router(d_model, n_experts, top_k, noise_std)
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(n_experts)])
    
    def forward(self, x: torch.Tensor) -> tuple:
        B, T, D = x.shape
        x_flat = x.view(-1, D)
        N = x_flat.shape[0]
        
        # Compute expert capacity
        capacity = int(self.capacity_factor * N * self.top_k / self.n_experts)
        capacity = max(capacity, 1)
        
        # Route
        gate_weights, expert_indices, router_logits = self.router(x_flat)
        
        output = torch.zeros_like(x_flat)
        tokens_dropped = 0
        expert_counts = torch.zeros(self.n_experts, dtype=torch.long)
        
        for k in range(self.top_k):
            for e_idx in range(self.n_experts):
                mask = (expert_indices[:, k] == e_idx)
                if mask.any():
                    token_indices = mask.nonzero(as_tuple=True)[0]
                    
                    # Apply capacity limit
                    current_count = expert_counts[e_idx].item()
                    remaining_capacity = max(0, capacity - current_count)
                    
                    if len(token_indices) > remaining_capacity:
                        # Drop tokens that exceed capacity
                        tokens_dropped += len(token_indices) - remaining_capacity
                        token_indices = token_indices[:remaining_capacity]
                    
                    if len(token_indices) > 0:
                        expert_counts[e_idx] += len(token_indices)
                        expert_input = x_flat[token_indices]
                        expert_output = self.experts[e_idx](expert_input)
                        w = gate_weights[token_indices, k].unsqueeze(-1)
                        output[token_indices] += w * expert_output
        
        # Dropped tokens use residual (identity) connection
        # This is handled by the residual connection in the transformer block
        
        info = {
            'tokens_dropped': tokens_dropped,
            'drop_rate': tokens_dropped / (N * self.top_k),
            'expert_counts': expert_counts,
            'capacity': capacity,
        }
        
        return output.view(B, T, D), router_logits, info


# Demo: token dropping with different capacity factors
x = torch.randn(4, 64, 256)  # 256 tokens total

for cf in [0.5, 1.0, 1.25, 1.5, 2.0]:
    moe_cap = MoELayerWithCapacity(256, 512, n_experts=8, top_k=2, capacity_factor=cf)
    moe_cap.train()
    out, _, info = moe_cap(x)
    print(f"capacity_factor={cf:.2f}: capacity={info['capacity']}, "
          f"dropped={info['tokens_dropped']}/{256*2} ({info['drop_rate']:.1%}), "
          f"expert_counts={info['expert_counts'].tolist()}")

## 6. Integrate MoE into a Transformer

In [ ]:
class MoETransformerBlock(nn.Module):
    """Transformer block with MoE FFN."""
    def __init__(self, d_model: int, n_heads: int, n_experts: int,
                 top_k: int = 2, d_ff: int = None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Standard attention
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        
        # MoE replaces the dense FFN
        self.moe = MoELayer(d_model, d_ff, n_experts, top_k)
    
    def forward(self, x, attn_mask=None):
        # Self-attention with residual
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, attn_mask=attn_mask)
        x = x + h
        
        # MoE FFN with residual
        h = self.norm2(x)
        moe_out, router_logits = self.moe(h)
        x = x + moe_out
        
        return x, router_logits


class MoETransformer(nn.Module):
    """Small transformer with MoE layers."""
    def __init__(self, vocab_size: int, d_model: int, n_heads: int,
                 n_layers: int, n_experts: int, top_k: int = 2,
                 max_len: int = 512):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([
            MoETransformerBlock(d_model, n_heads, n_experts, top_k)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, x):
        B, T = x.shape
        h = self.tok_emb(x) + self.pos_emb(torch.arange(T, device=x.device))
        
        all_router_logits = []
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        
        for layer in self.layers:
            h, router_logits = layer(h, attn_mask=mask)
            all_router_logits.append(router_logits)
        
        h = self.norm(h)
        logits = self.head(h)
        
        return logits, all_router_logits


# Compare dense vs MoE transformer
dense_model = MoETransformer(vocab_size=1000, d_model=256, n_heads=4,
                             n_layers=4, n_experts=1, top_k=1)  # 1 expert = dense
moe_model = MoETransformer(vocab_size=1000, d_model=256, n_heads=4,
                           n_layers=4, n_experts=8, top_k=2)

dense_params = sum(p.numel() for p in dense_model.parameters())
moe_params = sum(p.numel() for p in moe_model.parameters())

print(f"Dense transformer: {dense_params:,} params")
print(f"MoE transformer (8 experts, top-2): {moe_params:,} params")
print(f"Param ratio: {moe_params/dense_params:.1f}x")
print(f"\nBut active params per token are similar!")

# Estimate active params
# Each MoE layer: top_k experts active out of n_experts
# Active FFN params: (top_k / n_experts) * total_ffn_params
expert_params = sum(p.numel() for p in moe_model.layers[0].moe.experts[0].parameters())
active_ffn_per_layer = 2 * expert_params  # top-2
total_ffn_per_layer = 8 * expert_params
print(f"Per layer: {active_ffn_per_layer:,} active / {total_ffn_per_layer:,} total FFN params")

## 7. Training with Load Balancing Loss

In [ ]:
def train_moe_model(model, dataset, epochs=20, lr=1e-3, balance_coeff=0.01, label=""):
    """Train MoE model with auxiliary load balancing loss."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    losses = []
    balance_losses = []
    
    for epoch in range(epochs):
        perm = torch.randperm(len(dataset))
        epoch_loss = 0
        epoch_balance = 0
        n_batches = 0
        
        for i in range(0, len(dataset), 32):
            batch = dataset[perm[i:i+32]]
            inputs = batch[:, :-1]
            targets = batch[:, 1:]
            
            logits, all_router_logits = model(inputs)
            
            # Task loss
            task_loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), targets.reshape(-1)
            )
            
            # Load balancing loss (sum over all layers)
            bal_loss = torch.tensor(0.0)
            for router_logits in all_router_logits:
                router_probs = F.softmax(router_logits, dim=-1)
                _, expert_indices = torch.topk(router_probs, model.top_k, dim=-1)
                bal_loss = bal_loss + load_balancing_loss(
                    router_logits, expert_indices, model.n_experts, model.top_k
                )
            bal_loss = bal_loss / len(all_router_logits)
            
            # Total loss
            loss = task_loss + balance_coeff * bal_loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += task_loss.item()
            epoch_balance += bal_loss.item()
            n_batches += 1
        
        avg_loss = epoch_loss / n_batches
        avg_balance = epoch_balance / n_batches
        losses.append(avg_loss)
        balance_losses.append(avg_balance)
        
        if (epoch + 1) % 5 == 0:
            print(f"  [{label}] Epoch {epoch+1}: task_loss={avg_loss:.4f}, balance_loss={avg_balance:.4f}")
    
    return losses, balance_losses


# Create dataset
dataset = torch.randint(0, 1000, (500, 32))
# Inject pattern
for i in range(4, 32, 4):
    dataset[:, i] = (dataset[:, i-1] + 1) % 1000

# Train MoE model
print("=== MoE Training (8 experts, top-2) ===")
moe_model = MoETransformer(vocab_size=1000, d_model=128, n_heads=4,
                           n_layers=3, n_experts=8, top_k=2)
moe_losses, moe_bal_losses = train_moe_model(
    moe_model, dataset, epochs=20, balance_coeff=0.01, label="MoE"
)

# Train dense baseline
print("\n=== Dense Baseline ===")
dense_model = MoETransformer(vocab_size=1000, d_model=128, n_heads=4,
                             n_layers=3, n_experts=1, top_k=1)
dense_losses, _ = train_moe_model(
    dense_model, dataset, epochs=20, balance_coeff=0.0, label="Dense"
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training curves
dense_p = sum(p.numel() for p in dense_model.parameters())
moe_p = sum(p.numel() for p in moe_model.parameters())

axes[0].plot(dense_losses, 'b-o', label=f'Dense ({dense_p:,})', markersize=3)
axes[0].plot(moe_losses, 'r-s', label=f'MoE 8x ({moe_p:,})', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Task Loss')
axes[0].set_title('Task Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Balance loss
axes[1].plot(moe_bal_losses, 'g-o', markersize=3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Load Balancing Loss')
axes[1].set_title('MoE Load Balancing Loss')
axes[1].grid(True, alpha=0.3)

# Expert utilization after training
moe_model.eval()
with torch.no_grad():
    test_batch = dataset[:100]
    _, all_router_logits = moe_model(test_batch[:, :-1])

# Aggregate expert selection across all layers
expert_usage = torch.zeros(8)
for router_logits in all_router_logits:
    probs = F.softmax(router_logits, dim=-1)
    _, indices = torch.topk(probs, 2, dim=-1)
    for e in range(8):
        expert_usage[e] += (indices == e).sum().item()
expert_usage = expert_usage / expert_usage.sum()

axes[2].bar(range(8), expert_usage.numpy(), color='steelblue', edgecolor='white')
axes[2].axhline(y=1/8, color='red', linestyle='--', label='Ideal (uniform)')
axes[2].set_xlabel('Expert ID')
axes[2].set_ylabel('Fraction of Tokens')
axes[2].set_title('Expert Utilization After Training')
axes[2].legend()

plt.tight_layout()
plt.show()

## 8. Effect of Load Balancing Coefficient

In [ ]:
# Train with different balance coefficients
coefficients = [0.0, 0.001, 0.01, 0.1]
results = {}

for coeff in coefficients:
    m = MoETransformer(vocab_size=1000, d_model=128, n_heads=4,
                       n_layers=3, n_experts=8, top_k=2)
    losses, bal = train_moe_model(m, dataset, epochs=15, balance_coeff=coeff,
                                  label=f"coeff={coeff}")
    
    # Measure expert utilization
    m.eval()
    with torch.no_grad():
        _, router_logits_list = m(dataset[:100, :-1])
    
    usage = torch.zeros(8)
    for rl in router_logits_list:
        probs = F.softmax(rl, dim=-1)
        _, idx = torch.topk(probs, 2, dim=-1)
        for e in range(8):
            usage[e] += (idx == e).sum().item()
    usage = usage / usage.sum()
    
    results[coeff] = {'losses': losses, 'usage': usage}
    print(f"  coeff={coeff}: final_loss={losses[-1]:.4f}, "
          f"usage_std={usage.std().item():.4f}\n")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for coeff, data in results.items():
    axes[0].plot(data['losses'], '-o', label=f'coeff={coeff}', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Task Loss')
axes[0].set_title('Task Loss vs Balance Coefficient')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Expert utilization comparison
x = np.arange(8)
width = 0.2
for i, (coeff, data) in enumerate(results.items()):
    axes[1].bar(x + i * width, data['usage'].numpy(), width, label=f'coeff={coeff}')
axes[1].axhline(y=1/8, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Expert ID')
axes[1].set_ylabel('Token Fraction')
axes[1].set_title('Expert Utilization by Balance Coefficient')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Active vs Total Parameters

In [ ]:
# Scaling comparison: how MoE scales total vs active params
d_model = 256
d_ff = 4 * d_model
n_layers = 12

# Dense baseline FFN params per layer: 2 * d_model * d_ff
dense_ffn = 2 * d_model * d_ff

configs_scale = {
    'Dense (1 expert)': (1, 1),
    'MoE 4x Top-1': (4, 1),
    'MoE 8x Top-2': (8, 2),
    'MoE 16x Top-2': (16, 2),
    'MoE 64x Top-2': (64, 2),
    'MoE 128x Top-1': (128, 1),
}

total_params = []
active_params = []
labels = []

for name, (n_exp, top_k) in configs_scale.items():
    total = n_exp * dense_ffn * n_layers  # total FFN params
    active = top_k * dense_ffn * n_layers  # active FFN params per token
    total_params.append(total)
    active_params.append(active)
    labels.append(name)
    print(f"{name:>20s}: total={total/1e6:>7.1f}M, active={active/1e6:>7.1f}M, "
          f"active/total={active/total*100:.1f}%")

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(labels))
width = 0.35

bars1 = ax.bar(x - width/2, [p/1e6 for p in total_params], width,
               label='Total Params', color='#3498db')
bars2 = ax.bar(x + width/2, [p/1e6 for p in active_params], width,
               label='Active Params/Token', color='#2ecc71')

ax.set_xlabel('Configuration')
ax.set_ylabel('FFN Parameters (M)')
ax.set_title('Total vs Active Parameters (FFN only)')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.legend()
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 10. Expert Specialization Visualization

In [ ]:
# Visualize which experts different input patterns activate
# Use the trained MoE model

moe_model.eval()
n_tokens = 100
seq_len = 31

# Generate test data
test_data = torch.randint(0, 1000, (1, seq_len))

with torch.no_grad():
    _, router_logits_list = moe_model(test_data)

# Visualize routing decisions per position per layer
n_layers_vis = len(router_logits_list)
fig, axes = plt.subplots(n_layers_vis, 1, figsize=(14, 3 * n_layers_vis))

for layer_idx, router_logits in enumerate(router_logits_list):
    ax = axes[layer_idx] if n_layers_vis > 1 else axes
    
    probs = F.softmax(router_logits, dim=-1).squeeze(0)  # (seq_len, n_experts)
    
    im = ax.imshow(probs.T.numpy(), aspect='auto', cmap='YlOrRd', vmin=0)
    ax.set_xlabel('Token Position')
    ax.set_ylabel('Expert ID')
    ax.set_title(f'Layer {layer_idx}: Router Probabilities')
    ax.set_yticks(range(8))
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

## 11. Top-K Selection Comparison

In [ ]:
# Compare Top-1 vs Top-2 vs Top-4 routing
topk_results = {}

for top_k in [1, 2, 4]:
    m = MoETransformer(vocab_size=1000, d_model=128, n_heads=4,
                       n_layers=3, n_experts=8, top_k=top_k)
    params = sum(p.numel() for p in m.parameters())
    
    losses, _ = train_moe_model(m, dataset, epochs=15, balance_coeff=0.01,
                                 label=f"top-{top_k}")
    topk_results[top_k] = {'losses': losses, 'params': params}
    print(f"  top-{top_k}: params={params:,}, final_loss={losses[-1]:.4f}\n")

plt.figure(figsize=(10, 5))
for k, data in topk_results.items():
    plt.plot(data['losses'], '-o', label=f'Top-{k} ({data["params"]:,} params)', markersize=3)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Effect of Top-K on MoE Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Top-1 (Switch style): fastest inference, more tokens dropped")
print("Top-2 (Mixtral style): better quality, standard choice")
print("Top-4: even better quality but diminishing returns, more compute")

## 12. Summary: MoE Design Space

| Design Choice | Options | Tradeoff |
|--------------|---------|----------|
| **Number of experts** | 8-128+ | More experts = more capacity but harder to balance |
| **Top-K** | 1-4 | Higher K = better quality but more compute |
| **Capacity factor** | 1.0-1.5 | Higher = fewer drops but more wasted compute |
| **Balance loss coeff** | 0.001-0.1 | Too low = collapse, too high = hurts task loss |
| **Which layers** | All or alternating | Alternating dense+MoE reduces total params |
| **Expert size** | Same as dense FFN | Can be smaller per expert, larger total |

In [ ]:
# Real-world MoE configurations
print("=" * 80)
print(f"{'Model':>25s} {'Experts':>8s} {'Top-K':>6s} {'Total':>10s} {'Active':>10s} {'Ratio':>8s}")
print("=" * 80)

models = [
    ('Mixtral 8x7B', 8, 2, '47B', '13B', '28%'),
    ('Switch-C (1.6T)', 2048, 1, '1.6T', '~1B', '<0.1%'),
    ('GShard (600B)', 2048, 2, '600B', '~10B', '1.7%'),
    ('DeepSeek-V2', 160, 6, '236B', '21B', '8.9%'),
    ('Grok-1', 8, 2, '314B', '~86B', '27%'),
    ('DBRX', 16, 4, '132B', '36B', '27%'),
]

for name, n_exp, topk, total, active, ratio in models:
    print(f"{name:>25s} {n_exp:>8d} {topk:>6d} {total:>10s} {active:>10s} {ratio:>8s}")

print("=" * 80)

## Interview Notes

**"Why MoE?"**
- Scale model capacity (total params) without scaling compute (active params per token)
- Mixtral 8x7B: 47B total params but only 13B active per token -- competitive with dense 70B models
- Better compute efficiency: more knowledge stored per FLOP
- Key insight: not all tokens need the same processing -- let the model specialize

**"What is expert collapse?"**
- Without load balancing, the router learns to always select the same 1-2 experts
- Most experts get zero gradient updates and become useless
- Wastes most of the model capacity
- Prevention: auxiliary load balancing loss, noise in router, capacity limits
- Balance loss encourages equal token distribution across experts

**"Active vs total parameters?"**
- Total params: all expert weights + shared weights (attention, embeddings)
- Active params: top-K expert weights + shared weights (what runs per token)
- Mixtral example: 47B total, 13B active (top-2 out of 8 experts)
- MoE models need more memory (store all experts) but less compute per token
- Inference throughput depends on active params, not total

**"How does routing work?"**
- Simple learned linear layer: $g(x) = \text{softmax}(W_g x)$
- Top-K selection: pick K highest-probability experts
- Renormalize selected weights to sum to 1
- Output: weighted sum of expert outputs
- Router adds negligible parameters ($d_{\text{model}} \times n_{\text{experts}}$)

**"Challenges of MoE?"**
- Expert collapse (solved with load balancing loss)
- Token dropping with capacity limits (can hurt quality)
- Higher memory requirements (all experts must be in memory)
- Communication overhead in distributed training (expert parallelism)
- Harder to fine-tune than dense models
- Batch size sensitivity (routing statistics need enough tokens)